# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access the metadata (do not subscript or iterate)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### Identify Record Sets
We use `dataset.record_sets` to list each record set, referencing by `@id`.

In [ ]:
# List available record sets by @id and name
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    print("  Field @ids:")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    - {field['@id']} ({field.get('name', '')})")
    print()

# Select the first record set for demonstration
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"Sample records from record set {record_set_id}:")
    for record in dataset.records(record_set=record_set_id):
        print(record)
        break  # Print only first record for brevity

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

# Load records from each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for Record Set @id: {rs_id}")
    print(df.columns.tolist())
    print(df.head())

# Use the record_set_id selected previously for further steps
main_record_set_id = record_set_id
df_main = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We reference field columns via their `@id` where possible.

In [ ]:
# Example: Find a numeric field (e.g., GAD-7 total score) in the record set
# List available columns to identify likely numeric fields
print("Columns in main DataFrame:")
print(df_main.columns.tolist())

# Suppose we find a column with @id 'gad7_total_score' (update as per true schema)
numeric_field_id = None
for col in df_main.columns:
    if 'gad7' in col and ('score' in col or 'total' in col):
        numeric_field_id = col
        break

# If not found, fallback to another likely numeric field
if numeric_field_id is None and 'phq9_total_score' in df_main.columns:
    numeric_field_id = 'phq9_total_score'

# For demonstration, set a threshold
if numeric_field_id:
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field, e.g., 'gender' referenced by its @id
    group_field_id = None
    for col in df_main.columns:
        if 'gender' in col:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and, if possible, a comparison by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not df_main[numeric_field_id].isnull().all():
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Visualize by group field if available
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we:
- Loaded metadata and explored record sets and fields by their `@id`s using `mlcroissant`.
- Extracted and previewed data from each record set.
- Performed simple EDA including outlier filtering, normalization, and grouping by demographic variables using `@id` references.
- Visualized distributions and relationships within the dataset.

This workflow demonstrates best practices for FAIR and AI-ready data handling, and can be extended to deeper analysis or model development.